<a href="https://colab.research.google.com/github/GiammaBigFishEngineer/MaskArchitectureAnomaly_CourseProject/blob/main/eomt/STEP5_lighiting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

In [1]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git

%cd MaskArchitectureAnomaly_CourseProject/eomt

Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 478, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 478 (delta 34), reused 24 (delta 24), pack-reused 440 (from 2)
Receiving objects: 100% (478/478), 30.83 MiB | 27.50 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys # Allows importing from the 'eomt' folder
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')

In [ ]:
!python3 -m pip install -r requirements.txt

In [5]:
!pip uninstall -y wandb
!pip install -U wandb

import wandb
from lightning.pytorch.loggers import WandbLogger

wandb.login()

wandb_logger = WandbLogger(project="cityscapes_finetuning", # Crea (o seleziona, se esiste già) una cartella macro chiamata progetto sul tuo profilo online. Tutti i tentativi e i test correlati a questo lavoro verranno raggruppati qui dentro
                           name="finetune_selective_run", # Assegna un nome specifico a questa esecuzione (es. eomt_baseline_run). Ti permette di distinguere questo addestramento dai futuri test (magari con iperparametri diversi).
                           log_model = False # Impedisce a WandB di caricare i file pesanti dei pesi del modello (.ckpt o .bin) sui loro server cloud alla fine di ogni epoca. Salva direttamente sul tuo Google Drive tramite il ModelCheckpoint, risparmiando banda internet e spazio cloud gratuito su WandB.
                           )


Found existing installation: wandb 0.19.10
Uninstalling wandb-0.19.10:
  Successfully uninstalled wandb-0.19.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 24.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gianmaria-difro03 (gianmaria-difro03-politencico-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Setup

In [10]:
import yaml
import torch
import importlib
import logging
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import warnings

from lightning import seed_everything
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
from lightning.pytorch.callbacks import ModelCheckpoint

seed_everything(0, verbose=False)

device = 0  # TODO: change to the GPU you want to use
IGNORE_INDEX = 255
img_idx = 10  # TODO: change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/Cityscapes_Backup/"  # drive folder of the cityscapes val set

def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [13]:
state_dict_path = "/content/drive/MyDrive/eomt_coco.bin" # punta ai pesi pre-addestrati su COCO
config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

with open(config_path, "r") as f:
    config_cs = yaml.safe_load(f)

target_img_size = (640, 640)
num_classes_final = 19  # classi di cityscapes

data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=2,
    num_workers=2,
    check_empty_targets=True,
    img_size = target_img_size,
    **data_module_kwargs
)
data.setup()

In [ ]:
# PARTE NUOVA

In [14]:
# 1. Setup Network (FORZA query a 200 per compatibilità COCO)
net_cfg = config_cs["model"]["init_args"]["network"]
encoder_cfg = net_cfg["init_args"]["encoder"]

# Inizializza Encoder
enc_mod, enc_cls = encoder_cfg["class_path"].rsplit(".", 1)
encoder = getattr(importlib.import_module(enc_mod), enc_cls)(img_size=(640,640), **encoder_cfg.get("init_args", {}))

# Inizializza Network (EoMT)
net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
network = getattr(importlib.import_module(net_mod), net_cls)(
    masked_attn_enabled=True, #MODIFICATO
    num_classes=19, # Cityscapes
    encoder=encoder,
    num_q=200,      # Fondamentale: deve matchare i pesi COCO
    num_blocks=3
)

# 2. Setup Lightning Module
lit_mod, lit_cls = config_cs["model"]["class_path"].rsplit(".", 1)
# model = getattr(importlib.import_module(lit_mod), lit_cls)(
#     network=network,
#     img_size=(640,640),
#     num_classes=19,
#     ckpt_path=state_dict_path,
#     load_ckpt_class_head=False, # Reinizializza la testa da zero
#     lr=5e-5,
#     llrd=0.9,
#     llrd_l2_enabled=False,
#     lr_mult=1.0,
#     weight_decay=0.05,
#     poly_power=0.9,
#     warmup_steps=(0, 0),
#     attn_mask_annealing_enabled=False,
#     attn_mask_annealing_start_steps=None,
#     attn_mask_annealing_end_steps=None
# ).to(device)

model_kwargs_final = {k: v for k, v in config_cs["model"]["init_args"].items() if k != "network"}
model_kwargs_final.pop("num_classes", None)
model_kwargs_final.pop("img_size", None)

model = getattr(importlib.import_module(lit_mod), lit_cls)(
          img_size=target_img_size,
          num_classes=num_classes_final,
          network=network,
          **model_kwargs_final,
          ckpt_path=state_dict_path,          # Serve alla TUA classe per caricare il .bin iniziale
          load_ckpt_class_head=False
      ).to(device)

#print(model)




# 3. FREEZING SELETTIVO
for param in model.parameters():
    param.requires_grad = False

# Sblocca Testa + Ultimo Blocco Decoder
layers_to_train = ["class_head", #sblocco la testa --> Impara i nomi delle 19 classi di Cityscapes
                   "mask_head", #--> Impara le geometrie e i contorni degli oggetti stradali
                   "network.upscale", #--> Adatta la risoluzione spaziale alle immagini urbane
                   "network.q" # sblocca le query
                   ]
for name, param in model.named_parameters():
    if any(k in name for k in layers_to_train):
        param.requires_grad = True
        print(f"SBLOCCATO: {name}")

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


SBLOCCATO: network.q.weight
SBLOCCATO: network.class_head.weight
SBLOCCATO: network.class_head.bias
SBLOCCATO: network.mask_head.0.weight
SBLOCCATO: network.mask_head.0.bias
SBLOCCATO: network.mask_head.2.weight
SBLOCCATO: network.mask_head.2.bias
SBLOCCATO: network.mask_head.4.weight
SBLOCCATO: network.mask_head.4.bias
SBLOCCATO: network.upscale.0.conv1.weight
SBLOCCATO: network.upscale.0.conv1.bias
SBLOCCATO: network.upscale.0.conv2.weight
SBLOCCATO: network.upscale.0.norm.weight
SBLOCCATO: network.upscale.0.norm.bias
SBLOCCATO: network.upscale.1.conv1.weight
SBLOCCATO: network.upscale.1.conv1.bias
SBLOCCATO: network.upscale.1.conv2.weight
SBLOCCATO: network.upscale.1.norm.weight
SBLOCCATO: network.upscale.1.norm.bias


In [19]:
from lightning.pytorch.callbacks import ModelCheckpoint

# 1. Definiamo il Checkpoint Callback
checkpoint_callback = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/Modelli_EoMT", # Dove salvare
    filename="eomt-cityscapes-finetuning-head_complete-{epoch:02d}-{metrics/val_iou_all:.3f}",
    monitor="metrics/val_iou_all", # La metrica da monitorare
    mode="max",                    # Vogliamo massimizzare la IoU
    save_top_k=1,                  # Salva solo il file migliore in assoluto
    save_last = True,
    verbose=True                   # ??
)

current_max_epochs = 10
use_checkpoint = True ##########################################
resume_ckpt_path = "/content/drive/MyDrive/Modelli_EoMT/last.ckpt"


# 2. Configuriamo il Trainer includendo il callback e il logger
trainer = L.Trainer(
    max_epochs=current_max_epochs,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger,          # Fondamentale per vedere i grafici
    callbacks=[checkpoint_callback], # Colleghiamo il salvataggio automatico

    #limit_train_batches=0.01,      # Opzionale: per test rapidi
    #limit_val_batches=0.01,        # Valida solo sul 5%

    log_every_n_steps=10,
    default_root_dir="/content/drive/MyDrive/Modelli_EoMT/"
)

# 3. Avvio Training (GESTIONE DEL CHECKPOINT)
if use_checkpoint:
  print(f"Ripresa dell'addestramento dal checkpoint: {resume_ckpt_path}")
  trainer.fit(model, datamodule=data, ckpt_path=resume_ckpt_path)
else:
  trainer.fit(model, datamodule=data)

# 4. Dopo il training, stampa dove si trova il file migliore
print(f"Miglior modello salvato in: {checkpoint_callback.best_model_path}")
print(f"Miglior mIoU raggiunta: {checkpoint_callback.best_model_score:.4f}")



INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


Ripresa dell'addestramento dal checkpoint: /content/drive/MyDrive/Modelli_EoMT/last.ckpt


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/Modelli_EoMT exists and is not empty.
INFO: Restoring states from the checkpoint path at /content/drive/MyDrive/Modelli_EoMT/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/MyDrive/Modelli_EoMT/last.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loading `train_dataloader` to estimate number of stepping batches.
INFO:lightning.pytorch.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
INFO: 
  | Name      | Type                   | Params | Mode 
-------------------------------------------------------------
0 | network   | EoMT                   | 93.6 M | train
1 | criterion | MaskClassificationLoss | 0      | train
2 | metrics   | ModuleList             | 0 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: mIoU: 69.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 69.6
INFO: Epoch 4, global step 7435: 'metrics/val_iou_all' reached 0.69616 (best 0.69616), saving model to '/content/drive/MyDrive/Modelli_EoMT/eomt-cityscapes-finetuning-head_complete-epoch=04-metrics/val_iou_all=0.696.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 4, global step 7435: 'metrics/val_iou_all' reached 0.69616 (best 0.69616), saving model to '/content/drive/MyDrive/Modelli_EoMT/eomt-cityscapes-finetuning-head_complete-epoch=04-metrics/val_iou_all=0.696.ckpt' as top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO: mIoU: 70.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.5
INFO: Epoch 5, global step 8922: 'metrics/val_iou_all' reached 0.70515 (best 0.70515), saving model to '/content/drive/MyDrive/Modelli_EoMT/eomt-cityscapes-finetuning-head_complete-epoch=05-metrics/val_iou_all=0.705.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 5, global step 8922: 'metrics/val_iou_all' reached 0.70515 (best 0.70515), saving model to '/content/drive/MyDrive/Modelli_EoMT/eomt-cityscapes-finetuning-head_complete-epoch=05-metrics/val_iou_all=0.705.ckpt' as top 1
INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [20]:
import os
import torch
from functools import partial

# 1. Bypassiamo il blocco di sicurezza di PyTorch 2.6+
torch.load = partial(torch.load, weights_only=False)

# 2. Definiamo i percorsi dei file
checkpoint_path = "/content/drive/MyDrive/Modelli_EoMT/last.ckpt"  # Il tuo file sorgente
output_bin_path = "/content/drive/MyDrive/Modelli_EoMT/eomt_model_weights.bin"  # Il file .bin finale

print(f"Caricamento del checkpoint Lightning da: {checkpoint_path}...")

try:
    # 3. Carichiamo il checkpoint completo (contiene epoch, optimizers, state_dict, ecc.)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    # Estrarre lo state_dict puro
    lightning_state_dict = checkpoint["state_dict"]

    # 4. Pulizia del prefisso 'model.' (opzionale ma caldamente consigliata per compatibilità)
    # Se il tuo LightningModule conteneva l'architettura dentro 'self.model', i pesi avranno chiavi tipo 'model.encoder...'
    clean_state_dict = {}
    for key, value in lightning_state_dict.items():
        if key.startswith("model."):
            new_key = key.replace("model.", "", 1) # Rimuove solo il primo prefisso 'model.'
            clean_state_dict[new_key] = value
        else:
            clean_state_dict[key] = value

    # 5. Salvataggio in formato standard .bin
    print(f"Salvataggio dello state_dict in formato .bin...")
    torch.save(clean_state_dict, output_bin_path)

    print("-" * 50)
    print(f"✅ Conversione completata con successo!")
    print(f"📦 File generato: {output_bin_path}")
    print(f"📊 Numero di tensori salvati: {len(clean_state_dict)}")
    print("-" * 50)

except FileNotFoundError:
    print(f"❌ Errore: Non ho trovato nessun file in '{checkpoint_path}'. Controlla il percorso.")
except Exception as e:
    print(f"❌ Si è verificato un errore durante la conversione: {str(e)}")

Caricamento del checkpoint Lightning da: /content/drive/MyDrive/Modelli_EoMT/last.ckpt...
Salvataggio dello state_dict in formato .bin...
--------------------------------------------------
✅ Conversione completata con successo!
📦 File generato: /content/drive/MyDrive/Modelli_EoMT/eomt_model_weights.bin
📊 Numero di tensori salvati: 198
--------------------------------------------------


# Evaluation

In [22]:
!pip install gcn

In [25]:
def infer_semantic(img, target, model, target_size=(640, 640)):
    """
    Esegue l'inferenza semantica con sliding window e pulisce le dimensioni extra.
    """
    model.eval()

    # Se l'immagine ha una forma tipo [1, 3, 640, 640] presa dalla lista,
    # rimuoviamo il batch iniziale per dare al modello il formato [3, 640, 640] corretto
    if len(img.shape) == 4 and img.shape[0] == 1:
        img = img.squeeze(0)

    img = img.to(device)

    # Impostiamo la dimensione della finestra del modello
    model.window_size = target_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img]
        img_sizes = [img.shape[-2:]]

        # Divisione in finestre (sliding window)
        crops, origins = model.window_imgs_semantic(imgs)

        # Forward pass
        mask_logits_layers, class_logits_layers = model(crops)

        # Prendiamo l'output dell'ultimo blocco del decoder
        m_logits = F.interpolate(mask_logits_layers[-1], target_size, mode="bilinear")
        c_logits = class_logits_layers[-1]

        # Conversione query-mask -> pixel-logits
        crop_pixel_logits = model.to_per_pixel_logits_semantic(m_logits, c_logits)

        # Ricomposizione dell'immagine originale dalle finestre
        full_logits = model.revert_window_logits_semantic(crop_pixel_logits, origins, img_sizes)

        # Selezione della classe dominante per ogni pixel
        preds = full_logits[0].argmax(0).cpu().numpy()

    # Conversione del Ground Truth per il confronto
    target_pixel = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()

    # ASSICURIAMOCI CHE SIANO INTERI E SENZA DIMENSIONI "1" ORFANE
    return preds.squeeze().astype(np.int32), target_pixel.squeeze().astype(np.int32)




def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Gestione dell'immagine (se normalizzata o meno)
    img_np = img.permute(1, 2, 0).cpu().numpy()
    # Se i valori sono fuori dal range [0,1], normalizziamo per la visualizzazione
    if img_np.max() > 1.0 or img_np.min() < 0:
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

    axes[0].imshow(img_np) # Ora usiamo l'immagine processata
    axes[0].set_title("Original Image")

    # 2. Applica la colormap alla predizione
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Model Prediction")

    # 3. Applica la colormap al target (Ground Truth)
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Ground Truth")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [26]:
import numpy as np
import torch
import gcn
from tqdm import tqdm


# Funzioni di calcolo (rimangono invariate nella logica)
def fast_hist(a, b, n):
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    return np.diag(hist) / (np.maximum(1.0, hist.sum(1) + hist.sum(0) - np.diag(hist)))


# =====================================================================
# 1. SETUP DELLE MATRICI DI CONFUSIONE E DEI MODELLI
# =====================================================================
num_eval_classes = 19
val_loader = data.val_dataloader()

hist_finetuned = np.zeros((num_eval_classes, num_eval_classes))

# --- STEP A: VALUTAZIONE CON IL MODELLO FINE-TUNED ---
print(f"📊 Fase 1: Valutazione in corso con il modello FINE-TUNED su {len(val_loader.dataset)} immagini...")

# FONDAMENTALE: Assicuriamoci che tutto il modello sia sulla GPU corretta
model = model.to(device)
model.eval()

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Fine-tuned Evaluation"):
        imgs, targets = batch

        for j in range(len(imgs)):
            # Inferenza con il modello Fine-Tuned
            pred_ft, target_ft = infer_semantic(
                imgs[j],
                targets[j],
                model,
                target_size=target_img_size
            )
            hist_finetuned += fast_hist(target_ft.flatten(), pred_ft.flatten(), num_eval_classes)

# =====================================================================
# 2. CALCOLO DEI RISULTATI FINALI (IoU e mIoU)
# =====================================================================
ious_finetuned = per_class_iu(hist_finetuned)
miou_finetuned = np.nanmean(ious_finetuned)

# Classi ufficiali Cityscapes
classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

# =====================================================================
# 3. VISUALIZZAZIONE TABELLA COMPARATIVA CONSOLIDATA
# =====================================================================
print("\n" + "="*30)
print(f"{'CLASSE CITYSCAPES':<22} | {'IoU FINE-TUNED %':<15}")
print("-" * 30)

for name, iou_ft in zip(classes, ious_finetuned):
    ft_p = iou_ft * 100
    print(f"{name:<22} | {ft_p:>14.2f}%")

print("-" * 30)
print(f"{'MEAN IoU GLOBAL':<22} | {miou_finetuned*100:>14.2f}%")
print("="*30)

📊 Fase 1: Valutazione in corso con il modello FINE-TUNED su 500 immagini...



Fine-tuned Evaluation: 100%|██████████| 250/250 [03:15<00:00,  1.28it/s]


CLASSE CITYSCAPES      | IoU FINE-TUNED %
------------------------------
road                   |          97.58%
sidewalk               |          81.06%
building               |          91.75%
wall                   |          58.56%
fence                  |          54.53%
pole                   |          51.63%
traffic light          |          64.43%
traffic sign           |          71.18%
vegetation             |          91.24%
terrain                |          65.38%
sky                    |          94.29%
person                 |          75.33%
rider                  |          34.96%
car                    |          92.83%
truck                  |          71.00%
bus                    |          70.81%
train                  |          40.23%
motorcycle             |          48.02%
bicycle                |          74.81%
------------------------------
MEAN IoU GLOBAL        |          69.98%


In [ ]:
# sblocco head completa

import numpy as np
import torch
import gc
from tqdm import tqdm


# Funzioni di calcolo (rimangono invariate nella logica)
def fast_hist(a, b, n):
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    return np.diag(hist) / (np.maximum(1.0, hist.sum(1) + hist.sum(0) - np.diag(hist)))


# =====================================================================
# 1. SETUP DELLE MATRICI DI CONFUSIONE E DEI MODELLI
# =====================================================================
num_eval_classes = 19
val_loader = data.val_dataloader()

hist_finetuned = np.zeros((num_eval_classes, num_eval_classes))

# --- STEP A: VALUTAZIONE CON IL MODELLO FINE-TUNED ---
print(f"📊 Fase 1: Valutazione in corso con il modello FINE-TUNED su {len(val_loader.dataset)} immagini...")

# FONDAMENTALE: Assicuriamoci che tutto il modello sia sulla GPU corretta
model = model.to(device)
model.eval()

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Fine-tuned Evaluation"):
        imgs, targets = batch

        for j in range(len(imgs)):
            # Inferenza con il modello Fine-Tuned
            pred_ft, target_ft = infer_semantic(
                imgs[j],
                targets[j],
                model,
                target_size=target_img_size
            )
            hist_finetuned += fast_hist(target_ft.flatten(), pred_ft.flatten(), num_eval_classes)

# =====================================================================
# 2. CALCOLO DEI RISULTATI FINALI (IoU e mIoU)
# =====================================================================
ious_finetuned = per_class_iu(hist_finetuned)
miou_finetuned = np.nanmean(ious_finetuned)

# Classi ufficiali Cityscapes
classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

# =====================================================================
# 3. VISUALIZZAZIONE TABELLA COMPARATIVA CONSOLIDATA
# =====================================================================
print("\n" + "="*30)
print(f"{'CLASSE CITYSCAPES':<22} | {'IoU FINE-TUNED %':<15}")
print("-" * 30)

for name, iou_ft in zip(classes, ious_finetuned):
    ft_p = iou_ft * 100
    print(f"{name:<22} | {ft_p:>14.2f}%")

print("-" * 30)
print(f"{'MEAN IoU GLOBAL':<22} | {miou_finetuned*100:>14.2f}%")
print("="*30)

📊 Fase 1: Valutazione in corso con il modello FINE-TUNED su 500 immagini...


Fine-tuned Evaluation: 100%|██████████| 250/250 [03:06<00:00,  1.34it/s]


CLASSE CITYSCAPES      | IoU FINE-TUNED %
------------------------------
road                   |          97.67%
sidewalk               |          81.58%
building               |          91.93%
wall                   |          59.18%
fence                  |          55.06%
pole                   |          52.61%
traffic light          |          63.91%
traffic sign           |          71.76%
vegetation             |          91.37%
terrain                |          64.82%
sky                    |          94.44%
person                 |          75.20%
rider                  |          28.12%
car                    |          92.11%
truck                  |          71.40%
bus                    |          66.62%
train                  |          38.96%
motorcycle             |          45.14%
bicycle                |          74.63%
------------------------------
MEAN IoU GLOBAL        |          69.29%


In [ ]:
# import numpy as np
# from tqdm import tqdm

# # Funzioni di calcolo (rimangono invariate nella logica)
# def fast_hist(a, b, n):
#     k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
#     return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

# def per_class_iu(hist):
#     return np.diag(hist) / (np.maximum(1.0, hist.sum(1) + hist.sum(0) - np.diag(hist)))

# # 1. Preparazione
# num_eval_classes = 19
# hist = np.zeros((num_eval_classes, num_eval_classes))
# val_loader = data.val_dataloader()

# print(f"Inizio valutazione su {len(val_loader.dataset)} immagini...")

# # 2. Ciclo di valutazione
# model.eval()
# with torch.no_grad():
#     for batch in tqdm(val_loader):
#         imgs, targets = batch # In CityscapesSemantic solitamente sono tensori

#         # Iteriamo sul batch (es. batch_size=2)
#         for j in range(imgs.shape[0]):
#             # Eseguiamo l'inferenza (usando la versione corretta della funzione)
#             pred_array, target_array = infer_semantic(
#                 imgs[j],
#                 targets[j],
#                 model,
#                 target_size=target_img_size
#             )

#             # Aggiorniamo la matrice di confusione
#             hist += fast_hist(target_array.flatten(), pred_array.flatten(), num_eval_classes)

# # 3. Calcolo Risultati
# ious = per_class_iu(hist)
# miou = np.nanmean(ious)

# # Classi ufficiali Cityscapes
# classes = [
#     "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
#     "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
#     "truck", "bus", "train", "motorcycle", "bicycle"
# ]

# # 4. Visualizzazione Tabella
# print("\n" + "="*40)
# print(f"{'CLASSE':<20} | {'IoU %':<10}")
# print("-" * 35)
# for name, iou in zip(classes, ious):
#     print(f"{name:<20} | {iou*100:>8.2f}%")
# print("-" * 35)
# print(f"{'MEAN IoU':<20} | {miou*100:>8.2f}%")
# print("="*40)

In [ ]:
#parte vecchia

In [ ]:
# warnings.filterwarnings(
#     "ignore",
#     message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
# )

# # Load encoder
# encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]#
# encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)#
# encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)#
# encoder = encoder_cls(img_size=target_img_size, **encoder_cfg.get("init_args", {}))#

# # Load network
# network_cfg = current_config["model"]["init_args"]["network"]
# network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
# network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
# network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
# network = network_cls(
#     masked_attn_enabled=False,
#     num_classes=num_classes_final,
#     encoder=encoder,
#     **network_kwargs,
# )

# # Load Lightning module
# lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
# lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
# model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

# if "stuff_classes" in current_config["data"].get("init_args", {}):
#     model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

## Load pre-trained weights from Hugging Face Hub
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [ ]:
# # Ripuliamo i kwargs new
# model_kwargs_final = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}
# model_kwargs_final.pop("num_classes", None) # Rimuoviamo per non avere sovrascritture
# model_kwargs_final.pop("img_size", None)

# # FIX CRUCIALE: Rimuoviamo eventuali 'num_classes' dai kwargs che potrebbero sovrascrivere il nostro valore
# # if "num_classes" in model_kwargs_final:
# #   del model_kwargs_final["num_classes"]

# # new
# if "stuff_classes" in current_config["data"].get("init_args", {}):
#     model_kwargs_final["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

# name = current_config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
# is_dinov3 = "dinov3" in name if name else False

# if is_dinov3:
#     model_kwargs["ckpt_path"] = state_dict_path
#     model_kwargs["delta_weights"] = True

# model = (
#     lit_cls(
#         img_size=target_img_size,
#         num_classes=num_classes_final,
#         network=network,
#         **model_kwargs_final,
#     )
#     .to(device)
# )

## Transfer Learning and Fine Tuning

In [ ]:
# if not is_dinov3:
#     state_dict = torch.load(state_dict_path, map_location=f"cuda:{device}", weights_only=True)

#     # Freeze all, only unlock the head
#     keys_to_delete = [
#               "network.class_head.weight",
#               "network.class_head.bias",
#               "criterion.empty_weight"
#           ]
#     for key in keys_to_delete:
#         if key in state_dict:
#             del state_dict[key]

# model.load_state_dict(state_dict, strict=False)


In [ ]:
# for param in model.parameters():
#     param.requires_grad = False

# for param in model.network.class_head.parameters():
#     param.requires_grad = True

# # if hasattr(model.network, 'transformer_decoder'):
# #     for param in model.network.transformer_decoder.layers[-1].parameters():
# #         param.requires_grad = True

# # print(f"Parametri addestrabili: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# # checkpoint
# checkpoint_callback = ModelCheckpoint(
#   dirpath="/content/drive/MyDrive/CourseProjectAnomaly/checkpoints", # folder for saving
#   filename="coco-finetuned-cityscapes-{epoch:02d}", # file name
#   save_last=True, # always save last epoch
#   save_top_k=1,
#   monitor="metrics/val_iou_all",
#   mode="max"
# )

# # fine tuning dei parametri sbloccati
# trainer = L.Trainer(
#     max_epochs=5,              # 10 Numero epoche suggerito
#     accelerator="gpu",


#     limit_train_batches=0.05, # Usa solo il 5% dei dati
#     limit_val_batches=0.05,   # Valida solo sul 5%


#     devices=1,
#     precision="16-mixed",       # tip del progetto
#     callbacks=[checkpoint_callback],
#     log_every_n_steps=10,
#     #logger=wandb_logger,
#     default_root_dir="/content/drive/MyDrive/CourseProjectAnomaly/checkpoints"
# )

# # Avvio Fine-tuning
# trainer.fit(model, datamodule=data)

In [ ]:
# ### aggiunta per inviare i risultati visuali alla dashboard online

# # 1. Recupera un'immagine dal set di validazione
# img_val, target_val = data.val_dataloader().dataset[img_idx]

# # 2. Esegui l'inferenza usando la funzione definita prima
# pred_array, target_array = infer_semantic(img_val, target_val)

# # 3. Prepara le versioni colorate per la visualizzazione
# mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)
# pred_colored = apply_colormap(pred_array, mapping)
# target_colored = apply_colormap(target_array, mapping)

# # 4. Denormalizza l'immagine originale per WandB (se necessario)
# # Questo serve per evitare che l'immagine originale appaia troppo scura o strana
# img_input = img_val.permute(1, 2, 0).cpu().numpy()
# if img_input.max() > 1.0 or img_input.min() < 0:
#     img_input = (img_input - img_input.min()) / (img_input.max() - img_input.min())

# # 5. Logga finalmente su WandB
# wandb_logger.log_image(
#     key="final_results",
#     images=[img_input, pred_colored, target_colored],
#     caption=["Original Image", "Model Prediction", "Ground Truth"]
# )

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
# def infer_semantic(img, target):
#     model.eval()
#     model.to(device) # aggiunto per riportare in GPU
#     model.network.encoder.to(device) # aggiunto per riportare in GPU

#     # aggiunto Sposta l'immagine sul dispositivo del modello (GPU)
#     img = img.to(device)

#     # FIX  PER LE FINESTRE
#     # Forza il modello a usare la dimensione della finestra corretta
#     # (640 per COCO, 1024 per Cityscapes)
#     model.window_size = target_img_size[0]

#     with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
#         imgs = [img.to(device)]
#         img_sizes = [img.shape[-2:] for img in imgs]
#         crops, origins = model.window_imgs_semantic(imgs)

#         mask_logits_per_layer, class_logits_per_layer = model(crops)
#         mask_logits = F.interpolate(
#             mask_logits_per_layer[-1], target_img_size, mode="bilinear"
#         )

#         crop_logits = model.to_per_pixel_logits_semantic(
#             mask_logits, class_logits_per_layer[-1]
#         )
#         logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
#         preds = logits[0].argmax(0).cpu()

#     pred_array = preds.numpy()

#     target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()
#     return pred_array, target_array


# def plot_semantic_results(img, pred_array, target_array):
#     mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

#     fig, axes = plt.subplots(1, 3, figsize=(15, 5))

#     # Gestione dell'immagine (se normalizzata o meno)
#     img_np = img.permute(1, 2, 0).cpu().numpy()
#     # Se i valori sono fuori dal range [0,1], normalizziamo per la visualizzazione
#     if img_np.max() > 1.0 or img_np.min() < 0:
#         img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

#     axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
#     axes[0].set_title("Image")
#     axes[1].imshow(apply_colormap(pred_array, mapping))
#     axes[1].set_title("Prediction")
#     axes[2].imshow(apply_colormap(target_array, mapping))
#     axes[2].set_title("Target")

#     for ax in axes:
#         ax.axis("off")

#     plt.tight_layout()
#     plt.show()

In [ ]:
# # run for the cityscapes pre-trained model
# img_cs, target_cs = data.val_dataloader().dataset[img_idx]
# pred_array_cs, target_array_cs = infer_semantic(img_cs, target_cs)
# plot_semantic_results(img_cs, pred_array_cs, target_array_cs)

In [ ]:
# # run for the coco pre-trained model
# img_cc, target_cc = data.val_dataloader().dataset[img_idx]
# pred_array_cc, target_array_cc = infer_semantic(img_cc, target_cc)
# plot_semantic_results(img_cc, pred_array_cc, target_array_cc)

## Evaluation
Consistent evaluation strategy that allows fair comparison across both models for the semantic segmentation task

In [ ]:
# import numpy as np
# from tqdm import tqdm

# def fast_hist(a, b, n):
#     """
#     Calcola la matrice di confusione.
#     a: ground truth (target), b: predizione, n: numero di classi
#     """
#     k = (a >= 0) & (a < n) & (b >= 0) & (b < n) # Modifica qui: filtra anche b
#     return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

# def per_class_iu(hist):
#     """Calcola la IoU per ogni classe dalla matrice di confusione."""
#     return np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist))

# # setting
# num_eval_classes = 19 # Cityscapes ha 19 classi di valutazione
# hist = np.zeros((num_eval_classes, num_eval_classes))
# val_loader = data.val_dataloader()

# print(f"evaluation on {len(val_loader)} images...")
# print(f"Current model: {'COCO' if use_coco else 'Cityscapes'}")

# # evaluation, modificato per batch_size > 1
# for i, batch in enumerate(tqdm(val_loader)):
#     # Estraiamo le liste di immagini e target dal batch
#     # batch[0] è la lista delle immagini, batch[1] è la lista dei target
#     imgs_in_batch = batch[0]
#     targets_in_batch = batch[1]

#     # Iteriamo su ogni singolo elemento all'interno del batch (nel tuo caso, 2 iterazioni)
#     for j in range(len(imgs_in_batch)):
#         img = imgs_in_batch[j]
#         target = targets_in_batch[j]

#         # Eseguiamo l'inferenza semantica (include la mappatura se use_coco=True)
#         pred_array, target_array = infer_semantic(img, target)

#         # Aggiorniamo la matrice di confusione
#         hist += fast_hist(target_array.flatten(), pred_array.flatten(), num_eval_classes)

# # calcolo finale
# ious = per_class_iu(hist)
# miou = np.nanmean(ious)

# # visualization results
# classes = [
#     "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
#     "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
#     "truck", "bus", "train", "motorcycle", "bicycle"
# ]

# print("\n" + "="*30)
# print(f"Results MIOU - {'COCO' if use_coco else 'CITYSCAPES'}")
# print("="*30)
# for name, iou in zip(classes, ious):
#     print(f"{name:15}: {iou*100:6.2f}%")
# print("-" * 30)
# print(f"MEAN IoU: {miou*100:6.2f}%")
# print("="*30)